# 04- Evaluation Homework solutions:

### Q1. Generating questions

Generating questions for all 72 pages costs money and takes time, so let's start small and generate questions for just the first 3 pages:

01-agentic-rag/lessons/01-intro.md

01-agentic-rag/lessons/02-environment.md

01-agentic-rag/lessons/03-rag.md

Each call returns the token usage, which most LLM APIs report on the response object (e.g. response.usage.input_tokens / prompt_tokens).

What's the average number of input tokens across these 3 calls?

140
1400
14000
140000
These numbers vary between runs, even with the same model, so pick the closest option. A different provider or model may land further apart, but the input tokens stay in the same order of magnitude - the prompt we send is the same.

In [2]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

In [136]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [4]:
doc=documents[0]
doc['content']
doc['filename']

'01-agentic-rag/lessons/01-intro.md'

In [18]:
doc.keys()

dict_keys(['content', 'filename'])

In [5]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

In [7]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

In [8]:
import json

user_prompt = json.dumps(doc)

In [9]:
messages = [
    {"role": "developer", "content": data_gen_instructions},
    {"role": "user", "content": user_prompt}
]

In [10]:
response = openai_client.responses.parse(
    model="gpt-5.4-mini",
    input=messages,
    text_format=Questions
)

In [11]:
result = response.output_parsed

print(result)

questions=['What is a retrieval-augmented generation system supposed to do, and why would you use one instead of just asking the model directly?', 'Why is the course building the RAG example in plain Python instead of using a framework right away?', 'What are the main limits of large language models that make RAG useful?', 'What kind of example app is this module building from the FAQ data?', 'What will be covered in the first part of the module, and how is the second part different?']


In [12]:
print(result.questions)

['What is a retrieval-augmented generation system supposed to do, and why would you use one instead of just asking the model directly?', 'Why is the course building the RAG example in plain Python instead of using a framework right away?', 'What are the main limits of large language models that make RAG useful?', 'What kind of example app is this module building from the FAQ data?', 'What will be covered in the first part of the module, and how is the second part different?']


In [13]:
from evaluation_utils import llm_structured

In [14]:
result, usage = llm_structured(
    openai_client,
    data_gen_instructions,
    user_prompt,
    Questions
)

print(result.questions)

['What is an LLM, and why does it sometimes give answers that sound right but are actually wrong?', 'Why does this course treat the language model like a black box instead of digging into how it works?', 'How does retrieval-augmented generation help with problems like outdated knowledge or missing access to private data?', 'What are the main steps you build in the first part of this module to make the FAQ RAG system work?', 'What changes in the second part when the RAG pipeline becomes more agent-like?']


In [15]:
usage.input_tokens, usage.output_tokens

(1020, 116)

In [16]:
from evaluation_utils import calc_price

In [17]:
cost = calc_price(usage)

cost

{'input_cost': 0.0007650000000000001,
 'output_cost': 0.000522,
 'total_cost': 0.001287}

In [19]:
records = []

for q in result.questions:
    records.append({
        "question": q,
        "document": doc["filename"]
    })

records

[{'question': 'What is an LLM, and why does it sometimes give answers that sound right but are actually wrong?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does this course treat the language model like a black box instead of digging into how it works?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'How does retrieval-augmented generation help with problems like outdated knowledge or missing access to private data?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are the main steps you build in the first part of this module to make the FAQ RAG system work?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What changes in the second part when the RAG pipeline becomes more agent-like?',
  'document': '01-agentic-rag/lessons/01-intro.md'}]

In [24]:
doc_0=documents[0]
doc_1=documents[1]
doc_2=documents[2]
lst_docs=[doc_0,doc_1,doc_2]
avg_input_tokens=0
for i in lst_docs:
    result, usage = llm_structured(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )
    print(result.questions)
    avg_input_tokens+=usage.input_tokens/len(lst_docs)


['What is a RAG system doing differently from a plain LLM when it answers a question?', 'Why do LLMs sometimes give wrong answers or fail on recent info?', 'Why does the course use a provider API instead of running a model locally?', 'What are the main things the FAQ agent in this module is supposed to do?', 'What changes in the second part of the module to make the pipeline more agent-like?']
["What is Retrieval-Augmented Generation, and how does it help when an LLM doesn't know the answer from its training data?", 'Why do people use LLMs as black boxes in this course instead of training or inspecting a model directly?', 'What kinds of problems do LLMs have, like cutoff knowledge, missing access to private data, and hallucinations?', 'What are the main steps this module will cover to build a FAQ-style RAG system from scratch in Python?', 'How is the course project set up, and what changes in the later part when the system becomes more agent-like?']
['What is a retrieval-augmented gene

In [25]:
print(avg_input_tokens)

1020.0


### Q2. First result with text search
Take the first question from the ground truth:

q = ground_truth[0]["question"]
After running text_search for it, what's the filename of the first result?

01-agentic-rag/lessons/01-intro.md

01-agentic-rag/lessons/03-rag.md

01-agentic-rag/lessons/13-function-calling.md

01-agentic-rag/lessons/10-rag-next-steps.md

In [138]:
import pandas as pd 
ground_truth= pd.read_csv("ground-truth.csv")
ground_truth.head(2)

,question,filename
0,What exactly is a retrieval-augmented generati...,01-agentic-rag/lessons/01-intro.md
1,Why does this course build the RAG project in ...,01-agentic-rag/lessons/01-intro.md


In [140]:
ground_truth=ground_truth.to_dict(orient='records')

In [141]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [142]:
chunk_texts = [chunk['content'] for chunk in chunks]

In [143]:
from embedder import Embedder

embed = Embedder()

In [144]:
X = embed.encode_batch(chunk_texts)

In [146]:
# 1. Initialize and build the traditional keyword index
from minsearch import Index, VectorSearch

text_index = Index(text_fields=["content"], keyword_fields=["filename"])
text_index.fit(chunks)

In [147]:
def text_search(query,num_results=5):

    return text_index.search(
        query,
        num_results=5,
    )

In [149]:
q = ground_truth[0]["question"]

In [150]:
results = text_search(q)

In [151]:
results[0]['filename']

'01-agentic-rag/lessons/03-rag.md'

### Q3. First result with vector search
After running vector_search for the same question, what's the filename of the first result?

01-agentic-rag/lessons/01-intro.md

01-agentic-rag/lessons/03-rag.md

04-evaluation/lessons/11-evaluation-intro.md

04-evaluation/lessons/12-rag-answers.md

This question was generated from 01-agentic-rag/lessons/01-intro.md. Notice that one method finds the right page at the top and the other doesn't. That's exactly why we measure across the whole dataset instead of trusting one query.

In [152]:
vector_index = VectorSearch()
vector_index.fit(X, chunks)

In [153]:
def vector_search(query,num_results=5):
    # 1. Convert the raw text string query into a numerical vector array
    query_vector = embed.encode_batch([query])[0]
    
    # 2. Pass that vector array down to your vector index
    return vector_index.search(
        query_vector,
        num_results=5
    )

In [154]:
results = vector_search(q)

In [155]:
results[0]['filename']

'01-agentic-rag/lessons/01-intro.md'

### Q4. Evaluating text search

Evaluate text_search on the ground truth data.

What's the Hit Rate?

0.55
0.66
0.76
0.88

In [161]:
# 1. Metric and evaluation definitions matching the list-of-dictionaries format
def compute_relevance(ground_truth_record, search_results):
    """Checks if each retrieved document matches the expected target filename."""
    target_filename = ground_truth_record["filename"]
    return [1 if r["filename"] == target_filename else 0 for r in search_results]

def hit_rate(relevance_list):
    """Returns 1 if the target document hit anywhere in the candidate list."""
    return int(1 in relevance_list)

def mrr(relevance_list):
    """Calculates 1/rank for the first matched target document found."""
    for rank, r in enumerate(relevance_list):
        if r == 1:
            return 1 / (rank + 1)
    return 0

def evaluate(search_func, ground_truth_list):
    """Computes final system-wide Hit Rate and MRR metrics."""
    hit_rates = []
    mrrs = []
    for record in ground_truth_list:
        results = search_func(record["question"])
        relevance = compute_relevance(record, results)
        hit_rates.append(hit_rate(relevance))
        mrrs.append(mrr(relevance))
    return {
        "hit_rate": sum(hit_rates) / len(hit_rates),
        "mrr": sum(mrrs) / len(mrrs)
    }

# 2. RUN DIRECTLY (No conversion needed since ground_truth is already a list!)
print("Evaluating baseline text search...")
metrics = evaluate(text_search, ground_truth)

print(f"Hit Rate: {metrics['hit_rate']:.4f}")
print(f"MRR:      {metrics['mrr']:.4f}")

Evaluating baseline text search...
Hit Rate: 0.7583
MRR:      0.5943


### Q5. Evaluating vector search
Now evaluate vector_search - the part we left for the homework, since the module only evaluated keyword search.

What's the MRR?

0.35
0.45
0.55
0.65

In [163]:
# Evaluate the vector search baseline directly using your list of dicts
print("Evaluating baseline vector search...")
vector_metrics = evaluate(vector_search, ground_truth)

print(f"Hit Rate: {vector_metrics['hit_rate']:.4f}")
print(f"MRR:      {vector_metrics['mrr']:.4f}")

Evaluating baseline vector search...


Hit Rate: 0.7250
MRR:      0.5486


### Q6. Tuning hybrid search
The k constant in RRF controls how much the top ranks matter. A smaller k sharpens the gap between positions, so being at the top of a list counts for more. The RRF paper uses 60 as a default, but the best value depends on the data

so let's measure it.
Evaluate hybrid_search over the full ground truth dataset for k values 1, 50, 100, and 200. Compare the MRR values for these runs.

Which k gives the best MRR?

1
50
100
200
Several values of k may give the same MRR. If there's a tie, pick the smallest k.

In [164]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc["start"])
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]

In [165]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

In [166]:
for k in [1,50,100,200]:
    def hybrid_k(query,k=k):
        text_results=text_search(query,num_results=10)
        vector_results = vector_search(query, num_results=10)
        return rrf([text_results, vector_results], k=k)
    results= evaluate(hybrid_k,ground_truth)
    print(f"k={k:3d}-> Hit Rate: {results['hit_rate']:.4f},MRR:{results['mrr']:.4f}")

k=  1-> Hit Rate: 0.8417,MRR:0.6458
k= 50-> Hit Rate: 0.8417,MRR:0.6468
k=100-> Hit Rate: 0.8417,MRR:0.6468
k=200-> Hit Rate: 0.8417,MRR:0.6468
